## 1) Checking the GPU


In [ ]:
!nvidia-smi

## 2) Connecting to Google drive and find the source images


In [ ]:
from google.colab import drive
drive.mount('/content/drive') # Mounting Google dirve

IMAGES_DIR = "TODO" # Image path

import os
exts = ('.jpg', '.jpeg', '.png')
imgs = sorted(f for f in os.listdir(IMAGES_DIR) if f.lower().endswith(exts)) # Check if the input file is in the correct format
print(len(imgs), "images found; first 3:", imgs[:3])
assert len(imgs) > 0, "No images found - check IMAGES_DIR"

Mounted at /content/drive
10 images found; first 3: ['0001.jpg', '0002.jpg', '0003.jpg']


## 3) Download 3DGS

In [ ]:
%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
!ls /content/gaussian-splatting

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 603, done.
remote: Total 603 (delta 0), reused 0 (delta 0), pack-reused 603 (from 1)
Receiving objects: 100% (603/603), 2.09 MiB | 27.82 MiB/s, done.
Resolving deltas: 100% (346/346), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100% (174/174), done.        
remote: Total 3293 (delta 171), reused 280 (delta 148), pack-reused 2971 (from 1)        
Receiving objects:

In [ ]:
import shutil
DATA = "TODO" # For instance /content/data/myscene
os.makedirs(DATA + "/input", exist_ok=True)
for f in imgs:
    shutil.copy(os.path.join(IMAGES_DIR, f), DATA + "/input/")
print(len(os.listdir(DATA + "/input")), "files copied to", DATA + "/input")

10 files copied to /content/data/myscene/input


## 4) COLMAP: Checking cameras positions


In [ ]:
!apt-get -qq install -y colmap

Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../03-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1build1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../04-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.9build

In [ ]:
%cd /content/gaussian-splatting
import os
if not os.path.exists("convert.py"):
    !wget -q https://raw.githubusercontent.com/graphdeco-inria/gaussian-splatting/main/convert.py
!QT_QPA_PLATFORM=offscreen python3 convert.py -s /content/data/myscene --no_gpu

/content/gaussian-splatting
W20261011 16:52:40.169100  3430 feature_extraction.cc:403] Your current options use the maximum number of threads on the machine to extract features. Extracting SIFT features on the CPU can consume a lot of RAM per thread for large images. Consider reducing the maximum image size and/or the first octave or manually limit the number of extraction threads. Ignore this warning, if your machine has sufficient memory for the current settings.
I20261011 16:52:40.169389  3432 misc.cc:198] 
Feature extraction
I20261011 16:53:39.433678  3437 feature_extraction.cc:254] Processed file [1/10]
I20261011 16:53:39.433772  3437 feature_extraction.cc:257]   Name:            0002.jpg
I20261011 16:53:39.433794  3437 feature_extraction.cc:283]   Dimensions:      4080 x 3060
I20261011 16:53:39.433812  3437 feature_extraction.cc:286]   Camera:          #1 - OPENCV
I20261011 16:53:39.433830  3437 feature_extraction.cc:289]   Focal Length:    2914.29px (Prior)
I20261011 16:53:39.43

## 5) Checking the results of the COLMAP

In [ ]:
sparse = DATA + "/sparse/0"
assert os.path.exists(sparse + "/points3D.bin"), (
    "COLMAP failed: sparse/0 is missing. Photos probably have too little overlap, "
    "are blurry, or the subject has no texture. Retake photos; do not continue.")
registered = len(os.listdir(DATA + "/images"))
print(f"{registered} of {len(imgs)} images registered by COLMAP")
print(os.listdir(sparse))

10 of 10 images registered by COLMAP
['points3D.bin', 'cameras.bin', 'images.bin']


## 6) Installing Python 3.7

In [ ]:
!wget -O mini.sh https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
!chmod +x mini.sh
!bash ./mini.sh -b -f -p /usr/local
!conda install -q -y python=3.7
import sys
sys.path.append('/usr/local/lib/python3.7/site-packages')
!python --version  # Should say Python 3.7.x

--2026-10-11 16:59:39--  https://repo.anaconda.com/miniconda/Miniconda3-py37_23.1.0-1-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.32.241, 104.16.191.158, 2606:4700::6810:20f1, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.32.241|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 90665082 (86M) [application/x-sh]
Saving to: ‘mini.sh’

mini.sh             100%[===================>]  86.46M   160MB/s    in 0.5s    

2026-10-11 16:59:39 (160 MB/s) - ‘mini.sh’ saved [90665082/90665082]

PREFIX=/usr/local
Unpacking payload ...
                                                                                 
Installing base environment...





Preparing transaction: - \ | / done
Executing transaction: \ | / - \ | / - \ | / - \ | / - \ | done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the 

## 7) Download and install CUDA 11.8

In [ ]:
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!chmod +x cuda_11.8.0_520.61.05_linux.run
!./cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page
import os
os.environ['PATH'] += ':/usr/local/cuda-11.8/bin'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia'
!nvcc --version  # Should show CUDA 11.8

--2026-10-11 17:00:22--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 2.16.106.132, 2.16.106.152
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|2.16.106.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4336730777 (4.0G) [application/octet-stream]
Saving to: ‘cuda_11.8.0_520.61.05_linux.run’

cuda_11.8.0_520.61. 100%[===================>]   4.04G   252MB/s    in 19s     

2026-10-11 17:00:42 (217 MB/s) - ‘cuda_11.8.0_520.61.05_linux.run’ saved [4336730777/4336730777]

 Failed to verify gcc version. See log at /var/log/cuda-installer.log for details.
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


## 8) Installing PyTorch

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

Found existing installation: torch 1.12.1+cu116
Uninstalling torch-1.12.1+cu116:
  Successfully uninstalled torch-1.12.1+cu116
Found existing installation: torchvision 0.13.1+cu116
Uninstalling torchvision-0.13.1+cu116:
  Successfully uninstalled torchvision-0.13.1+cu116
Found existing installation: torchaudio 0.12.1+cu116
Uninstalling torchaudio-0.12.1+cu116:
  Successfully uninstalled torchaudio-0.12.1+cu116
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116
  Using cached https://download-r2.pytorch.org/whl/cu116/torch-1.12.1%2Bcu116-cp37-cp37m-linux_x86_64.whl (1904.8 MB)
  Using cached https://download-r2.pytorch.org/whl/cu116/torchvision-0.13.1%2Bcu116-cp37-cp37m-linux_x86_64.whl (23.5 MB)
  Using cached https://download-r2.pytorch.org/whl/cu116/torchaudio-0.12.1%2Bcu116-cp37-cp37m-linux_x86_64.whl (3.8 MB)


## 9) installing CUDA 3 modules regarding to 3DGS

In [ ]:
!pip install -q plyfile
%cd /content/gaussian-splatting
!pip install -q /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q /content/gaussian-splatting/submodules/simple-knn

/content/gaussian-splatting
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


## 10) Training the model

In [ ]:
%cd /content/gaussian-splatting
!python train.py -s /content/data/myscene \
    -m /content/drive/MyDrive/gs_output/myscene \
    --iterations 30000 \
    --save_iterations 7000 15000 30000 \
    --checkpoint_iterations 7000 15000

/content/gaussian-splatting
Optimizing /content/drive/MyDrive/gs_output/myscene
Output folder: /content/drive/MyDrive/gs_output/myscene [11/10 17:24:29]
Tensorboard not available: not logging progress [11/10 17:24:29]
Reading camera 10/10 [11/10 17:24:30]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [11/10 17:24:30]
Loading Training Cameras [11/10 17:24:30]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [11/10 17:24:30]
Loading Test Cameras [11/10 17:24:39]
Number of points at initialisation :  3216 [11/10 17:24:39]
Training progress:  23% 7000/30000 [16:14<1:13:38,  5.21it/s, Loss=0.0499598]
[ITER 7000] Evaluating train: L1 0.021191127225756647 PSNR 28.270512771606448 [11/10 17:40:53]

[ITER 7000] Saving Gaussians [11/10 17:40:53]

[ITER 7000] Saving Checkpoint [11/10 17:41:12]
Training progress:  50% 15000/30000 [45:14<56:15,  4.44it

## 11) Creating a PLY file


In [ ]:
PLY = "/content/drive/MyDrive/gs_output/myscene/point_cloud/iteration_30000/point_cloud.ply"
!ls -lh {PLY}
print("PLY is saved on your Google Drive at:", PLY)

-rw------- 1 root root 502M Oct 11 19:07 /content/drive/MyDrive/gs_output/myscene/point_cloud/iteration_30000/point_cloud.ply
PLY is saved on your Google Drive at: /content/drive/MyDrive/gs_output/myscene/point_cloud/iteration_30000/point_cloud.ply


## *In case of problem with CUDA version, run this

In [ ]:
# import os
#
# # Miniconda (step 7) left an old libstdc++ in /usr/local/lib that shadows the system one
# # (the CUDA installer crashed with 'GLIBCXX_3.4.30 not found'). Make the system lib win.
# os.environ["LD_LIBRARY_PATH"] = "/lib/x86_64-linux-gnu:/usr/local/cuda-11.8/lib64:/usr/lib64-nvidia"
#
# # gcc 13 is too new for CUDA 11.8 -> gcc-11 as default
# !apt-get -qq install -y gcc-11 g++-11
# !update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-11 110 --slave /usr/bin/g++ g++ /usr/bin/g++-11
# !gcc --version | head -1
#
# # CUDA 11.8 toolkit (skipped if already installed)
# if not os.path.exists("/usr/local/cuda-11.8/bin/nvcc"):
#     !rm -f /var/log/cuda-installer.log
#     !wget -nc -q https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
#     !bash cuda_11.8.0_520.61.05_linux.run --silent --toolkit --no-drm --no-man-page --override
# if not os.path.exists("/usr/local/cuda-11.8/bin/nvcc"):
#     !tail -n 25 /var/log/cuda-installer.log
#     raise RuntimeError("CUDA 11.8 install failed - send the log printed above")
#
# # PREPEND so 11.8 wins over Colab's own CUDA 13
# os.environ["CUDA_HOME"] = "/usr/local/cuda-11.8"
# os.environ["PATH"] = "/usr/local/cuda-11.8/bin:" + os.environ["PATH"]
# os.environ["CC"], os.environ["CXX"] = "gcc-11", "g++-11"
# os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"   # Tesla T4
# !which nvcc
# !nvcc --version | tail -2   # must say release 11.8

gcc (Ubuntu 11.5.0-1ubuntu1~24.04.1) 11.5.0
/usr/local/cuda-11.8/bin/nvcc
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0
